# Approach 3 — Hybrid: Pre-Screen + ChatTS-14B + Embedded Stats

**Thesis Section: Chapter 4.3 (Best performing approach)**

This is the complete Approach 3 pipeline:
1. Statistical pre-screener determines anomaly type and selects chunk
2. Signal statistics embedded in the prompt context
3. ChatTS-14B classifies using the appropriate MCQ template
4. Category extracted with regex-based parser

**Requires**: A100 80GB GPU, ChatTS-14B checkpoint

In [ ]:
import sys
sys.path.insert(0, '../..')

from dotenv import load_dotenv
load_dotenv('../../.env')

import os
import torch
import numpy as np

from src.data.timeseer_client import fetch_series_api, list_series_api, AREAS
from src.data.ground_truth import GROUND_TRUTH, LABEL_NAMES
from src.models.chatts_loader import load_model
from src.prescreener.analyze import hybrid_analyze
from src.inference.chatts_infer import ask_chatts_chunk
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table, save_results

print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load ChatTS-14B
model, tokenizer, processor = load_model('ChatTS-14B')
ACTIVE_MODEL = 'ChatTS-14B'

In [ ]:
# Run Approach 3 on a single area
AREA = 'Reactor 1'

tags    = list_series_api(AREA)
results = []

for tag in tags:
    print(f'\n── {tag} ──')
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        print('  FAILED')
        continue

    chunk, question, tname, detected, chunk_desc = hybrid_analyze(vals, idx, tag)
    print(f'  Pre-screen → {detected}')
    print(f'  Template   → {tname}')
    print(f'  Chunk      → {chunk_desc}')

    torch.cuda.empty_cache()
    answer = ask_chatts_chunk(chunk, question, model=model,
                              tokenizer=tokenizer, processor=processor)
    cat_code, cat_label = extract_category(answer, detected)

    gt = GROUND_TRUTH.get(tag, '?')
    results.append({
        'Tag':      tag,
        'gt':       gt,
        'Detected': ', '.join(detected),
        'Template': tname,
        'Category': cat_code,
        'Label':    cat_label,
        'Answer':   answer[:400],
    })
    print(f'  ChatTS → {cat_code}) {cat_label}  (GT: {gt})')

In [ ]:
# Summary
print_summary_table(results, title=f'APPROACH 3 — {AREA}', model_name=ACTIVE_MODEL)

preds = [r['Category'] for r in results if r['gt'] != '?']
gts   = [r['gt']       for r in results if r['gt'] != '?']
metrics = compute_metrics(preds, gts)
print(f'\nAccuracy: {metrics["accuracy"]:.3f} ({metrics["n_correct"]}/{metrics["n_samples"]})')

save_results(results, f'../../data/chatts14b_approach3_{AREA.replace(" ","_")}.txt',
             header=f'ChatTS Approach 3 — {AREA} | {ACTIVE_MODEL}')